# Mejorar viviendas

En este notebook se pretende hacer lo siguiente:


1. Recoger las descripciones de las viviendas de los anuncios.
2. Pasar un MLL para extraer más información de las descripciones.
3. Enriquecer el dataset actual.

## Dependencias

In [1]:
# Librerías generales
import pandas as pd
import numpy as np

from urllib.request import urlopen
from bs4 import BeautifulSoup

import re

In [2]:
# Cargamos Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ruta = '/content/drive/MyDrive/Colab Notebooks/TFM/AñadimosDescripciones/pisos_descripcion.csv'
ruta = '/content/drive/MyDrive/Colab Notebooks/TFM/AñadimosDescripciones/pisos_con_descripción_filtrado.csv'

pisos = pd.read_csv(ruta, sep=';')

In [ ]:
pisos.head()

,tipo,precio,localidad,latitud,longitud,superficie_construida,habitaciones,banyos,antiguedad,conservacion,...,orientacion,garaje,piscina,aire_acondicionado,trastero,ascensor,planta,fecha,detalle_url,Descripcion
0,Piso,139000,Ibi,38.624174,-0.574839,120.0,4.0,1.0,Entre 30 y 50 años,En buen estado,...,Este,Desconocido,Sin piscina,False,True,True,4ª,2026-02-24,https://www.pisos.com/comprar/piso-ibi_centro_...,En el municipio de ibi se encuentra este ampli...
1,Piso,200000,La Vila Joiosa Villajoyosa,38.502493,-0.241090,85.0,3.0,2.0,Entre 10 y 20 años,En buen estado,...,Desconocido,Desconocido,Sin piscina,True,True,True,0ª,2026-02-24,https://www.pisos.com/comprar/piso-montiboli_p...,"Piso en planta baja, superficie 85 m². Cuenta ..."
2,Ático,490000,Alicante Alacant,38.372956,-0.418654,125.0,3.0,2.0,Entre 10 y 20 años,A estrenar,...,Todos,Privado,Exterior,True,False,True,10ª,2026-02-02,https://www.pisos.com/comprar/atico-playas_pla...,"Superf. 125 m², 3 habitaciones ( 1 suite, 2 ..."
3,Piso,193700,Finestrat,38.528848,-0.165879,70.0,2.0,1.0,Entre 30 y 50 años,En buen estado,...,Desconocido,Privado,Exterior,False,False,True,1ª,2026-02-24,https://www.pisos.com/comprar/piso-finestrat_c...,En La Cala de Finestrat se encuentra este enca...
4,Chalet,298000,Sax,38.525305,-0.815885,531.0,5.0,2.0,Entre 10 y 20 años,En buen estado,...,Desconocido,Privado,Sin piscina,False,False,False,0ª,2025-12-11,https://www.pisos.com/comprar/chalet_rustico-s...,"Espectacular chalet independiente en Sax, con ..."


## Recolección de las descripciones

Primero vamos a añadir las descripciones de los anuncios a su respectiva vivienda:

In [ ]:
df = pisos.copy()

# Creamos la columna con Desconocida primero:
# df['Descripcion'] = 'Desconocida' lo comentamos porque sino sobreescribe lo que llevamos cuando volvemos a ejecutar

df.head()

,tipo,precio,localidad,latitud,longitud,superficie_construida,habitaciones,banyos,antiguedad,conservacion,...,orientacion,garaje,piscina,aire_acondicionado,trastero,ascensor,planta,fecha,detalle_url,Descripcion
0,Ático,70000,Alcoi Alcoy,38.700241,-0.481300,90.0,2.0,1.0,Entre 5 y 10 años,En buen estado,...,Desconocido,Desconocido,Sin piscina,False,True,False,5ª,2025-12-23,https://www.pisos.com/comprar/atico-alcoi_alco...,Desconocida_Revisada
1,Piso,139000,Ibi,38.624174,-0.574839,120.0,4.0,1.0,Entre 30 y 50 años,En buen estado,...,Este,Desconocido,Sin piscina,False,True,True,4ª,2026-02-24,https://www.pisos.com/comprar/piso-ibi_centro_...,En el municipio de ibi se encuentra este ampli...
2,Ático,950000,Alicante Alacant,38.380730,-0.410119,100.0,3.0,2.0,Desconocida,A reformar,...,Sur-Este,Privado,Comunitaria,False,False,True,10ª,2026-02-02,https://www.pisos.com/comprar/atico-playas_pla...,Desconocida_Revisada
3,Piso,200000,La Vila Joiosa Villajoyosa,38.502493,-0.241090,85.0,3.0,2.0,Entre 10 y 20 años,En buen estado,...,Desconocido,Desconocido,Sin piscina,True,True,True,0ª,2026-02-24,https://www.pisos.com/comprar/piso-montiboli_p...,"Piso en planta baja, superficie 85 m². Cuenta ..."
4,Piso,115000,Alcoi Alcoy,38.691936,-0.488745,90.0,4.0,2.0,Desconocida,En buen estado,...,Desconocido,Privado,Sin piscina,False,True,True,0ª,NaN,https://www.pisos.com/comprar/piso-alcoi_alcoy...,Desconocida_Revisada


In [ ]:
df['detalle_url'].iloc[1]

'https://www.pisos.com/comprar/piso-ibi_centro_urbano-58089976_443300/'

### Prueba

Vamos a hacer una prueba. Este url: https://www.pisos.com/comprar/piso-ibi_centro_urbano-58089976_443300/. Nos debería de devolver esta descrpición:

En el municipio de ibi se encuentra este amplio piso en venta, situado en la planta 4ª y con una superficie total de 120 m². El piso cuenta con 2 habitaciones individuales, 2 habitaciones dobles, 1 baño, balcón amplio, calefacción de calor azul, cocina reformada y muy amplia, comedor espacioso y luminoso con balcón, armarios empotrados, ascensor, suelo de terrazo y trastero en la misma vivienda más otro amplio en la terraza. El estado de conservación es bueno y está amueblado. La orientación es este, lo que garantiza una vivienda soleada. Además, el edificio dispone de terraza comunitaria y el nivel energético es clase e, con emisiones de 39 y consumo de 195. ¡No dejes pasar esta oportunidad de adquirir este acogedor piso en ibi! En pvp no incluidos los gastos de escrituración, registro, gestoría, impuestos, notaria y gestión de la inmobiliaria

In [ ]:
url = 'https://www.pisos.com/comprar/piso-ibi_centro_urbano-58089976_443300/'

html = urlopen(url, timeout=30)
soup = BeautifulSoup(html, "html.parser")

# Buscamos el div por su clase específica
div_descripcion = soup.find("div", class_="description__content")

if div_descripcion:
    # Extraemos el texto y eliminamos los espacios y saltos de línea sobrantes
    texto_descripcion = div_descripcion.text.strip()
    print(texto_descripcion)
else:
    print("No se encontró la descripción.")

En el municipio de ibi se encuentra este amplio piso en venta, situado en la planta 4ª y con una superficie total de 120 m². El piso cuenta con 2 habitaciones individuales, 2 habitaciones dobles, 1 baño, balcón amplio, calefacción de calor azul, cocina reformada y muy amplia, comedor espacioso y luminoso con balcón, armarios empotrados, ascensor, suelo de terrazo y trastero en la misma vivienda más otro amplio en la terraza. El estado de conservación es bueno y está amueblado. La orientación es este, lo que garantiza una vivienda soleada. Además, el edificio dispone de terraza comunitaria y el nivel energético es clase e, con emisiones de 39 y consumo de 195. ¡No dejes pasar esta oportunidad de adquirir este acogedor piso en ibi! En pvp no incluidos los gastos de escrituración, registro, gestoría, impuestos, notaria y gestión de la inmobiliaria


### Proceso

Ahora generalizamos el proceso para todas las filas del data frame:

In [ ]:
from urllib.request import urlopen
from bs4 import BeautifulSoup

def obtener_descripcion(url):
    """
    Conecta a la URL y extrae el texto del div 'description__content'.
    Si hay un error o no lo encuentra, devuelve None.
    """
    try:
        html = urlopen(url, timeout=30)
        soup = BeautifulSoup(html, "html.parser")

        div_descripcion = soup.find("div", class_="description__content")

        if div_descripcion:
            return div_descripcion.text.strip()
        return None

    except Exception as e:
        print(f"Error al acceder a {url}: {e}")
        return None

In [ ]:
def actualizar_descripciones(df):
    """
    Recorre el DataFrame buscando descripciones 'Desconocida',
    las extrae de la web y guarda el progreso en el archivo CSV.
    """
    actualizadas = 0  # contador de actualizaciones

    for idx in df.index:
        if df.loc[idx, "Descripcion"] == "Desconocida":

            url = df.loc[idx, "detalle_url"]
            descripcion = obtener_descripcion(url)

            if descripcion:
                df.loc[idx, "Descripcion"] = descripcion
                actualizadas += 1

                # Guardamos el progreso tras cada acierto
                df.to_csv(ruta,
                          index=False,
                          sep=";",
                          encoding="utf-8")

                print(f"Fila actualizada: {idx} | CSV sobrescrito con éxito | Total actualizadas: {actualizadas}")

            else:
                # Si falla la descarga o no existe el div, marcamos para no reintentar infinitamente
                df.loc[idx, "Descripcion"] = "Desconocida_Revisada"

    print(f"\nTotal descripciones actualizadas correctamente: {actualizadas}")
    return df

In [ ]:
# actualizar_descripciones(df)

Fila actualizada: 35269 | CSV sobrescrito con éxito | Total actualizadas: 1
Fila actualizada: 35272 | CSV sobrescrito con éxito | Total actualizadas: 2
Fila actualizada: 35282 | CSV sobrescrito con éxito | Total actualizadas: 3
Fila actualizada: 35286 | CSV sobrescrito con éxito | Total actualizadas: 4
Fila actualizada: 35292 | CSV sobrescrito con éxito | Total actualizadas: 5
Fila actualizada: 35293 | CSV sobrescrito con éxito | Total actualizadas: 6
Fila actualizada: 35294 | CSV sobrescrito con éxito | Total actualizadas: 7
Fila actualizada: 35296 | CSV sobrescrito con éxito | Total actualizadas: 8
Fila actualizada: 35298 | CSV sobrescrito con éxito | Total actualizadas: 9
Fila actualizada: 35306 | CSV sobrescrito con éxito | Total actualizadas: 10
Fila actualizada: 35307 | CSV sobrescrito con éxito | Total actualizadas: 11
Fila actualizada: 35310 | CSV sobrescrito con éxito | Total actualizadas: 12
Fila actualizada: 35311 | CSV sobrescrito con éxito | Total actualizadas: 13
Fila act

,tipo,precio,localidad,latitud,longitud,superficie_construida,habitaciones,banyos,antiguedad,conservacion,...,orientacion,garaje,piscina,aire_acondicionado,trastero,ascensor,planta,fecha,detalle_url,Descripcion
0,Ático,70000,Alcoi Alcoy,38.700241,-0.481300,90.0,2.0,1.0,Entre 5 y 10 años,En buen estado,...,Desconocido,Desconocido,Sin piscina,False,True,False,5ª,2025-12-23,https://www.pisos.com/comprar/atico-alcoi_alco...,Desconocida_Revisada
1,Piso,139000,Ibi,38.624174,-0.574839,120.0,4.0,1.0,Entre 30 y 50 años,En buen estado,...,Este,Desconocido,Sin piscina,False,True,True,4ª,2026-02-24,https://www.pisos.com/comprar/piso-ibi_centro_...,En el municipio de ibi se encuentra este ampli...
2,Ático,950000,Alicante Alacant,38.380730,-0.410119,100.0,3.0,2.0,Desconocida,A reformar,...,Sur-Este,Privado,Comunitaria,False,False,True,10ª,2026-02-02,https://www.pisos.com/comprar/atico-playas_pla...,Desconocida_Revisada
3,Piso,200000,La Vila Joiosa Villajoyosa,38.502493,-0.241090,85.0,3.0,2.0,Entre 10 y 20 años,En buen estado,...,Desconocido,Desconocido,Sin piscina,True,True,True,0ª,2026-02-24,https://www.pisos.com/comprar/piso-montiboli_p...,"Piso en planta baja, superficie 85 m². Cuenta ..."
4,Piso,115000,Alcoi Alcoy,38.691936,-0.488745,90.0,4.0,2.0,Desconocida,En buen estado,...,Desconocido,Privado,Sin piscina,False,True,True,0ª,NaN,https://www.pisos.com/comprar/piso-alcoi_alcoy...,Desconocida_Revisada
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38009,Piso,95000,Elx Elche,38.269481,-0.686137,65.0,3.0,1.0,Más de 50 años,Reformado,...,Sur-Este,Desconocido,Sin piscina,True,False,False,1ª,2025-11-14,https://www.pisos.com/comprar/piso-elche_elx_c...,Desconocida_Revisada
38010,Piso,94000,Villena,38.635965,-0.867072,130.0,3.0,2.0,Entre 20 y 30 años,Reformado,...,Sur,Desconocido,Sin piscina,False,True,True,1ª,2026-02-13,https://www.pisos.com/comprar/piso-villena_cen...,Desconocida_Revisada
38011,Casa,150000,Callosa D'en Sarrià,38.650000,-0.123545,229.0,10.0,2.0,Más de 50 años,A reformar,...,Desconocido,Desconocido,Sin piscina,False,False,False,0ª,NaN,https://www.pisos.com/comprar/casa_rustica-cal...,Desconocida_Revisada
38012,Chalet,800000,La Nucia,38.611688,-0.127596,430.0,4.0,2.0,Entre 30 y 50 años,En buen estado,...,Desconocido,Privado,Sin piscina,False,False,False,0ª,2025-04-23,https://www.pisos.com/comprar/chalet_unifamili...,Venta chalet en parcela edificable en la nucia...


### Exportación del nuevo dataset

Una vez terminada la recolección, creamos un nuevo dataset llamado pisos_con_descripción_filtrado, donde nos quedamos solamente con las filas que tienen descripción, este número no coincide con el número total de filas debido a que muchos anuncios ya han seido borrados de la web:

In [ ]:
pisos_con_descripción_filtrado = pisos[pisos['Descripcion'] != 'Desconocida_Revisada']

pisos_con_descripción_filtrado.to_csv("pisos_con_descripción_filtrado.csv", sep=";", index=False)

## Enriquecimiento de los datos

Durante esta sección vamos a revisar mediante la descripción de cada anuncio con distintos modelos de LLM, las siguientes columnas:

* Antigüedad.
* Conservación.
* Terraza.
* Jardín.
* Garaje.
* Piscina.
* Aire acondicionado.
* Trastero.
* Ascensor.

### Qwen2.5-7B

#### Descarga del modelo

In [ ]:
# =========================
# 0. INSTALAR (solo si falta)
# =========================
import importlib

def install_if_missing(package):
    if importlib.util.find_spec(package) is None:
        !pip install -q {package}

install_if_missing("transformers")
install_if_missing("accelerate")
install_if_missing("bitsandbytes")
install_if_missing("huggingface_hub")

# =========================
# 1. MONTAR DRIVE
# =========================
from google.colab import drive
drive.mount('/content/drive')

# =========================
# 2. CONFIGURACIÓN
# =========================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-7B-Instruct"

save_path = "/content/drive/MyDrive/Colab Notebooks/TFM/AñadimosDescripciones/qwen_model"

# =========================
# 3. CUANTIZACIÓN
# =========================
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# =========================
# 4. DESCARGAR MODELO
# =========================
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=quantization_config
)

# =========================
# 5. GUARDAR EN DRIVE
# =========================
tokenizer.save_pretrained(save_path)
model.save_pretrained(save_path)

print("✅ Modelo guardado en:", save_path)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.8 MB/s eta 0:00:00
Mounted at /content/drive


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Modelo guardado en: /content/drive/MyDrive/Colab Notebooks/TFM/AñadimosDescripciones/qwen_model


#### Carga del modelo

In [4]:
# =========================
# 0. INSTALAR (solo si falta)
# =========================
import importlib

def install_if_missing(package):
    if importlib.util.find_spec(package) is None:
        !pip install -q {package}

install_if_missing("transformers")
install_if_missing("accelerate")
install_if_missing("bitsandbytes")

# =========================
# 1. MONTAR DRIVE
# =========================
from google.colab import drive
drive.mount('/content/drive')

# =========================
# 2. CONFIGURACIÓN
# =========================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

load_path = "/content/drive/MyDrive/Colab Notebooks/TFM/AñadimosDescripciones/qwen_model"

# =========================
# 3. CUANTIZACIÓN
# =========================
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# =========================
# 4. CARGAR DESDE DRIVE
# =========================
tokenizer = AutoTokenizer.from_pretrained(load_path)

model = AutoModelForCausalLM.from_pretrained(
    load_path,
    device_map="auto",
    quantization_config=quantization_config
)

print("🚀 Modelo cargado desde Drive")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.6 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:271: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

🚀 Modelo cargado desde Drive


#### Conservación

In [ ]:
pisos_origen = pisos.copy()
pisos_enriquecidos = pisos.copy()

Primero vamos a enriquecer la columna Conservación, preparamos el prompt:

In [ ]:
# 1. Definimos la función que habla con el modelo
def clasifica_conservacion_llm(descripcion):

    prompt = (
        "Analiza la siguiente descripción de una vivienda y clasifica el tipo de conservación que tiene. "
        "Responde estrictamente con una de estas 5 opciones:\n"
        "- 'A estrenar' (si la vivienda es nueva)\n"
        "- 'En buen estado' (si la vivienda no presenta defectos, no está reformada y tampoco es nueva)\n"
        "- 'Reformado' (si la vivienda ha sido reformada)\n"
        "- 'A reformar' (si la vivienda necesita ser reformada)\n"
        "- 'Desconocida Revisada' (si el tipo de conservación no es ninguno de los anteriores)"
        f"Descripción: {descripcion}"
    )
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    with torch.no_grad(): # Esto ahorra memoria GPU al procesar
        generated_ids = model.generate(**model_inputs, max_new_tokens=10, pad_token_id=tokenizer.eos_token_id)

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    resultado = response.split("assistant\n")[-1].strip().upper()
    # print(resultado)
    # Limpieza rápida por si el modelo añade un punto final o mayúsculas/minúsculas
    resultado_clean = resultado.lower()
    if "a estrenar" in resultado_clean:
        return "A estrenar"
    elif "reformado" in resultado_clean:
        return "Reformado"
    elif "a reformar" in resultado_clean:
        return "A reformar"
    elif "en buen estado" in resultado_clean:
        return "En buen estado"
    else:
        return "Desconocida Revisada"

Preparamos el código para recorrer todas las filas con conservación desconocida:

In [ ]:
import pandas as pd
import torch
import os

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

ruta = "/content/drive/MyDrive/Colab Notebooks/TFM/AñadimosDescripciones/DatasetsEnriquecidos/Qwen/pisos_enriquecidos_Qwen_co.csv"

if os.path.exists(ruta):
    print("El archivo ya existe")
    pisos_enriquecidos_Qwen_co = pd.read_csv(ruta, sep=";")
else:
    print("El archivo aún no existe")
    pisos_enriquecidos_Qwen_co = pisos_enriquecidos

# =========================
# 3. PROCESAMIENTO
# =========================
indices = pisos_enriquecidos_Qwen_co[pisos_enriquecidos_Qwen_co["conservacion"] == "Desconocida"].index

print(f"Filas a procesar: {len(indices)}")
guardar_cada = 100

for i, idx in enumerate(indices):
    descripcion = pisos_enriquecidos_Qwen_co.loc[idx, "Descripcion"]

    try:
        nueva_clasificacion = clasifica_conservacion_llm(descripcion)
        pisos_enriquecidos_Qwen_co.loc[idx, "conservacion"] = nueva_clasificacion

    except Exception as e:
        print(f"Error en fila {idx}: {e}")
        continue

    # =========================
    # 4. GUARDADO ITERATIVO
    # =========================
    if i % guardar_cada == 0:
      pisos_enriquecidos_Qwen_co.to_csv(ruta, sep=";", index=False)

    if i % guardar_cada == 0:
        print(f"Procesadas {i}/{len(indices)}")

pisos_enriquecidos_Qwen_co.to_csv(ruta, sep=";", index=False)

print("✅ Proceso terminado")

El archivo ya existe
Filas a procesar: 0
✅ Proceso terminado


In [ ]:
pisos_origen['conservacion'].value_counts()

,count
conservacion,
Desconocida,10107
En buen estado,1268
A estrenar,427
A reformar,129
Reformado,101


In [ ]:
pisos_enriquecidos_Qwen_co['conservacion'].value_counts()

,count
conservacion,
A estrenar,6273
En buen estado,4311
Reformado,933
A reformar,299
Desconocida Revisada,216


#### Terraza

Cargamos el anterior dataset:

In [ ]:
pisos_enriquecidos_Qwen_co = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/TFM/AñadimosDescripciones/DatasetsEnriquecidos/Qwen/pisos_enriquecidos_Qwen_co.csv', sep=';')

print(pisos_enriquecidos_Qwen_co.duplicated().sum())

pisos_enriquecidos_Qwen_co['terraza'].value_counts()

0


,count
terraza,
False,6066
True,5966


Preparamos la nueva función con el nuevo prompt:

In [ ]:
# 1. Definimos la función que habla con el modelo
def clasifica_terraza_llm(descripcion):

    prompt = (
        "Analiza la siguiente descripción de una vivienda y clasifica el tipo de terraza que tiene. "
        "Responde estrictamente con una de estas 2 opciones:\n"
        "- 'True' (si la vivienda tiene algún tipo de terraza)\n"
        "- 'False' (si la vivienda no tiene ningún tipo de terraza)\n"
        f"Descripción: {descripcion}"
    )
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    with torch.no_grad(): # Esto ahorra memoria GPU al procesar
        generated_ids = model.generate(**model_inputs, max_new_tokens=10, pad_token_id=tokenizer.eos_token_id)

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    resultado = response.split("assistant\n")[-1].strip().upper()
    # print(resultado)
    # Limpieza rápida por si el modelo añade un punto final o mayúsculas/minúsculas
    resultado_clean = resultado.lower()
    if "true" in resultado_clean:
        return "True"
    elif "false" in resultado_clean:
        return "False Revisada"
    else:
        return "Desconocida Revisada"

Preparamos el codigo para recorrer todas las filas con terraza:

In [ ]:
import pandas as pd
import torch
import os

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

ruta = "/content/drive/MyDrive/Colab Notebooks/TFM/AñadimosDescripciones/DatasetsEnriquecidos/Qwen/pisos_enriquecidos_Qwen_co_te.csv"

if os.path.exists(ruta):
    print("El archivo ya existe")
    pisos_enriquecidos_Qwen_co_te = pd.read_csv(ruta, sep=";")
else:
    print("El archivo aún no existe")
    pisos_enriquecidos_Qwen_co_te = pisos_enriquecidos_Qwen_co

# =========================
# 3. PROCESAMIENTO
# =========================
indices = pisos_enriquecidos_Qwen_co_te[pisos_enriquecidos_Qwen_co_te["terraza"] == False].index

print(f"Filas a procesar: {len(indices)}")
guardar_cada = 100

for i, idx in enumerate(indices):
    descripcion = pisos_enriquecidos_Qwen_co_te.loc[idx, "Descripcion"]

    try:
        nueva_clasificacion = clasifica_terraza_llm(descripcion)
        pisos_enriquecidos_Qwen_co_te.loc[idx, "terraza"] = nueva_clasificacion

    except Exception as e:
        print(f"Error en fila {idx}: {e}")
        continue

    # =========================
    # 4. GUARDADO ITERATIVO
    # =========================
    if i % guardar_cada == 0:
      pisos_enriquecidos_Qwen_co_te.to_csv(ruta, sep=";", index=False)

    if i % guardar_cada == 0:
        print(f"Procesadas {i}/{len(indices)}")

pisos_enriquecidos_Qwen_co_te.to_csv(ruta, sep=";", index=False)
print("✅ Proceso terminado")

El archivo ya existe
Filas a procesar: 6066
Error en fila 8: Torch not compiled with CUDA enabled
Error en fila 13: Torch not compiled with CUDA enabled
Error en fila 14: Torch not compiled with CUDA enabled
Error en fila 15: Torch not compiled with CUDA enabled
Error en fila 19: Torch not compiled with CUDA enabled
Error en fila 37: Torch not compiled with CUDA enabled
Error en fila 40: Torch not compiled with CUDA enabled
Error en fila 42: Torch not compiled with CUDA enabled
Error en fila 44: Torch not compiled with CUDA enabled
Error en fila 45: Torch not compiled with CUDA enabled
Error en fila 46: Torch not compiled with CUDA enabled
Error en fila 48: Torch not compiled with CUDA enabled
Error en fila 49: Torch not compiled with CUDA enabled
Error en fila 50: Torch not compiled with CUDA enabled
Error en fila 110: Torch not compiled with CUDA enabled
Error en fila 122: Torch not compiled with CUDA enabled
Error en fila 176: Torch not compiled with CUDA enabled
Error en fila 178: 

KeyboardInterrupt: 

In [ ]:
pisos_enriquecidos_Qwen_co['terraza'].value_counts()

,count
terraza,
False,6066
True,5966


In [ ]:
pisos_enriquecidos_Qwen_co_te = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/TFM/AñadimosDescripciones/DatasetsEnriquecidos/Qwen/pisos_enriquecidos_Qwen_co_te.csv', sep=';')

pisos_enriquecidos_Qwen_co_te['terraza'].value_counts()

,count
terraza,
True,6070
False Revisada,5962


#### Jardín

Cargamos el dataset:

In [ ]:
pisos_enriquecidos_Qwen_co_te = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/TFM/AñadimosDescripciones/DatasetsEnriquecidos/Qwen/pisos_enriquecidos_Qwen_co_te.csv', sep=';')

Preparamos el prompt:

In [ ]:
# 1. Definimos la función que habla con el modelo
def clasifica_jardin_llm(descripcion):

    prompt = (
        "Analiza la siguiente descripción de una vivienda y clasifica el tipo de jardin que tiene. "
        "Responde estrictamente con una de estas 3 opciones:\n"
        "- 'Privado' (si la vivienda tiene jardín privado)\n"
        "- 'Comunitario' (si la vivienda tiene jardín comunitario)\n"
        "- 'Sin Jardín' (si la vivienda no tiene jardín)\n"
        f"Descripción: {descripcion}"
    )
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    with torch.no_grad(): # Esto ahorra memoria GPU al procesar
        generated_ids = model.generate(**model_inputs, max_new_tokens=10, pad_token_id=tokenizer.eos_token_id)

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    resultado = response.split("assistant\n")[-1].strip().upper()
    # print(resultado)
    # Limpieza rápida por si el modelo añade un punto final o mayúsculas/minúsculas
    resultado_clean = resultado.lower()
    if "privado" in resultado_clean:
        return "Privado"
    elif "comunitario" in resultado_clean:
        return "Comunitario"
    elif "sin jardin" in resultado_clean:
        return "Sin Jardín"
    else:
        return "Desconocida Revisada"

Preparamos el codigo para recorrer todas las filas con terraza:

In [ ]:
import pandas as pd
import torch
import os

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

ruta = "/content/drive/MyDrive/Colab Notebooks/TFM/AñadimosDescripciones/DatasetsEnriquecidos/Qwen/pisos_enriquecidos_Qwen_co_te_ja.csv"

if os.path.exists(ruta):
    print("El archivo ya existe")
    pisos_enriquecidos_Qwen_co_te_ja = pd.read_csv(ruta, sep=";")
else:
    print("El archivo aún no existe")
    pisos_enriquecidos_Qwen_co_te_ja = pisos_enriquecidos_Qwen_co_te

# =========================
# 3. PROCESAMIENTO
# =========================
indices = pisos_enriquecidos_Qwen_co_te_ja[pisos_enriquecidos_Qwen_co_te_ja["jardin"] == "Desconocido"].index

print(f"Filas a procesar: {len(indices)}")
guardar_cada = 100

for i, idx in enumerate(indices):
    descripcion = pisos_enriquecidos_Qwen_co_te_ja.loc[idx, "Descripcion"]

    try:
        nueva_clasificacion = clasifica_jardin_llm(descripcion)
        pisos_enriquecidos_Qwen_co_te_ja.loc[idx, "jardin"] = nueva_clasificacion

    except Exception as e:
        print(f"Error en fila {idx}: {e}")
        continue

    # =========================
    # 4. GUARDADO ITERATIVO
    # =========================
    if i % guardar_cada == 0:
      pisos_enriquecidos_Qwen_co_te_ja.to_csv(ruta, sep=";", index=False)

    if i % guardar_cada == 0:
        print(f"Procesadas {i}/{len(indices)}")

pisos_enriquecidos_Qwen_co_te_ja.to_csv(ruta, sep=";", index=False)
print("✅ Proceso terminado")

El archivo ya existe
Filas a procesar: 66
Procesadas 0/66
✅ Proceso terminado


In [ ]:
pisos_enriquecidos_Qwen_co_te['jardin'].value_counts()

,count
jardin,
False,5909
Privado,3145
Desconocido,2769
Comunitario,209


In [ ]:
pisos_enriquecidos_Qwen_co_te_ja['jardin'].value_counts()

,count
jardin,
False,5909
Privado,3987
Desconocida Revisada,1608
Comunitario,528


#### Garaje

Cargamos el dataset anterior:

In [ ]:
pisos_enriquecidos_Qwen_co_te_ja = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/TFM/AñadimosDescripciones/DatasetsEnriquecidos/Qwen/pisos_enriquecidos_Qwen_co_te_ja.csv', sep=';')

Preparamos el prompt:

In [ ]:
# 1. Definimos la función que habla con el modelo
def clasifica_garaje_llm(descripcion):

    prompt = (
        "Analiza la siguiente descripción de una vivienda y clasifica el tipo de garaje que tiene. "
        "Responde estrictamente con una de estas 4 opciones:\n"
        "- 'Privado' (si la vivienda tiene garaje privado)\n"
        "- 'Comunitario' (si la vivienda tiene garaje comunitario)\n"
        "- 'Opcional' (si en la vivienda existe la opción de garaje)\n"
        "- 'Sin garaje' (si la vivienda no tiene garaje)\n"
        f"Descripción: {descripcion}"
    )
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    with torch.no_grad(): # Esto ahorra memoria GPU al procesar
        generated_ids = model.generate(**model_inputs, max_new_tokens=10, pad_token_id=tokenizer.eos_token_id)

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    resultado = response.split("assistant\n")[-1].strip().upper()
    # print(resultado)
    # Limpieza rápida por si el modelo añade un punto final o mayúsculas/minúsculas
    resultado_clean = resultado.lower()
    if "privado" in resultado_clean:
        return "Privado"
    elif "comunitario" in resultado_clean:
        return "Comunitario"
    elif "opcional" in resultado_clean:
        return "Opcional"
    elif "sin garaje" in resultado_clean:
        return "Sin Garaje"
    else:
        return "Desconocida Revisada"

Preparamos el codigo para recorrer todas las filas con garaje:

In [ ]:
import pandas as pd
import torch

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

ruta = "/content/drive/MyDrive/Colab Notebooks/TFM/AñadimosDescripciones/DatasetsEnriquecidos/Qwen/pisos_enriquecidos_Qwen_co_te_ja_ga.csv"

if os.path.exists(ruta):
    print("El archivo ya existe")
    pisos_enriquecidos_Qwen_co_te_ja_ga = pd.read_csv(ruta, sep=";")
else:
    print("El archivo aún no existe")
    pisos_enriquecidos_Qwen_co_te_ja_ga = pisos_enriquecidos_Qwen_co_te_ja

# =========================
# 3. PROCESAMIENTO
# =========================
indices = pisos_enriquecidos_Qwen_co_te_ja_ga[pisos_enriquecidos_Qwen_co_te_ja_ga["garaje"] == "Desconocido"].index

print(f"Filas a procesar: {len(indices)}")
guardar_cada = 100

for i, idx in enumerate(indices):
    descripcion = pisos_enriquecidos_Qwen_co_te_ja_ga.loc[idx, "Descripcion"]

    try:
        nueva_clasificacion = clasifica_garaje_llm(descripcion)
        pisos_enriquecidos_Qwen_co_te_ja_ga.loc[idx, "garaje"] = nueva_clasificacion

    except Exception as e:
        print(f"Error en fila {idx}: {e}")
        continue

    # =========================
    # 4. GUARDADO ITERATIVO
    # =========================
    if i % guardar_cada == 0:
      pisos_enriquecidos_Qwen_co_te_ja_ga.to_csv(ruta, sep=";", index=False)

    if i % guardar_cada == 0:
        print(f"Procesadas {i}/{len(indices)}")

pisos_enriquecidos_Qwen_co_te_ja_ga.to_csv(ruta, sep=";", index=False)
print("✅ Proceso terminado")

El archivo aún no existe
Filas a procesar: 3906
Procesadas 0/3906
Procesadas 100/3906
Procesadas 200/3906
Procesadas 300/3906
Procesadas 400/3906
Procesadas 500/3906
Procesadas 600/3906
Procesadas 700/3906
Procesadas 800/3906
Procesadas 900/3906
Procesadas 1000/3906
Procesadas 1100/3906
Procesadas 1200/3906
Procesadas 1300/3906
Procesadas 1400/3906
Procesadas 1500/3906
Procesadas 1600/3906
Procesadas 1700/3906
Procesadas 1800/3906
Procesadas 1900/3906
Procesadas 2000/3906
Procesadas 2100/3906
Procesadas 2200/3906
Procesadas 2300/3906
Procesadas 2400/3906
Procesadas 2500/3906
Procesadas 2600/3906
Procesadas 2700/3906
Procesadas 2800/3906
Procesadas 2900/3906
Procesadas 3000/3906
Procesadas 3100/3906
Procesadas 3200/3906
Procesadas 3300/3906
Procesadas 3400/3906
Procesadas 3500/3906
Procesadas 3600/3906
Procesadas 3700/3906
Procesadas 3800/3906
Procesadas 3900/3906
✅ Proceso terminado


In [ ]:
pisos_enriquecidos_Qwen_co_te_ja['garaje'].value_counts()

,count
garaje,
False,5909
Desconocido,3906
Privado,2216
Opcional,1


In [ ]:
pisos_enriquecidos_Qwen_co_te_ja_ga['garaje'].value_counts()

,count
garaje,
False,5909
Privado,3373
Sin Garaje,2416
Opcional,241
Desconocida Revisada,53
Comunitario,40


#### Piscina

Cargamos el dataset anterior:

In [ ]:
pisos_enriquecidos_Qwen_co_te_ja_ga = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/TFM/AñadimosDescripciones/DatasetsEnriquecidos/Qwen/pisos_enriquecidos_Qwen_co_te_ja_ga.csv', sep=';')

Preparamos el prompt:

In [ ]:
# 1. Definimos la función que habla con el modelo
def clasifica_piscina_llm(descripcion):

    prompt = (
        "Analiza la siguiente descripción de una vivienda y clasifica el tipo de piscina que tiene. "
        "Responde estrictamente con una de estas 4 opciones:\n"
        "- 'Privada' (si la vivienda tiene piscina privada)\n"
        "- 'Comunitaria' (si la vivienda tiene piscina comunitaria)\n"
        "- 'Climatizada' (si la vivienda tiene piscina climatizada)\n"
        "- 'Sin piscina' (si la vivienda no tiene piscina)\n"
        f"Descripción: {descripcion}"
    )
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    with torch.no_grad(): # Esto ahorra memoria GPU al procesar
        generated_ids = model.generate(**model_inputs, max_new_tokens=10, pad_token_id=tokenizer.eos_token_id)

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    resultado = response.split("assistant\n")[-1].strip().upper()
    # print(resultado)
    # Limpieza rápida por si el modelo añade un punto final o mayúsculas/minúsculas
    resultado_clean = resultado.lower()
    if "privada" in resultado_clean:
        return "Privada"
    elif "comunitaria" in resultado_clean:
        return "Comunitaria"
    elif "climatizada" in resultado_clean:
        return "Climatizada"
    elif "sin piscina" in resultado_clean:
        return "Sin Piscina"
    else:
        return "Desconocida Revisada"

Preparamos el codigo para recorrer todas las filas con piscina:

In [ ]:
import pandas as pd
import torch

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

ruta = "/content/drive/MyDrive/Colab Notebooks/TFM/AñadimosDescripciones/DatasetsEnriquecidos/Qwen/pisos_enriquecidos_Qwen_co_te_ja_ga_pi.csv"

if os.path.exists(ruta):
    print("El archivo ya existe")
    pisos_enriquecidos_Qwen_co_te_ja_ga_pi = pd.read_csv(ruta, sep=";")
else:
    print("El archivo aún no existe")
    pisos_enriquecidos_Qwen_co_te_ja_ga_pi = pisos_enriquecidos_Qwen_co_te_ja_ga

# =========================
# 3. PROCESAMIENTO
# =========================
indices = pisos_enriquecidos_Qwen_co_te_ja_ga_pi[pisos_enriquecidos_Qwen_co_te_ja_ga_pi["piscina"] == "Sin piscina"].index

print(f"Filas a procesar: {len(indices)}")
guardar_cada = 100

for i, idx in enumerate(indices):
    descripcion = pisos_enriquecidos_Qwen_co_te_ja_ga_pi.loc[idx, "Descripcion"]

    try:
        nueva_clasificacion = clasifica_piscina_llm(descripcion)
        pisos_enriquecidos_Qwen_co_te_ja_ga_pi.loc[idx, "piscina"] = nueva_clasificacion

    except Exception as e:
        print(f"Error en fila {idx}: {e}")
        continue

    # =========================
    # 4. GUARDADO ITERATIVO
    # =========================
    if i % guardar_cada == 0:
      pisos_enriquecidos_Qwen_co_te_ja_ga_pi.to_csv(ruta, sep=";", index=False)

    if i % guardar_cada == 0:
        print(f"Procesadas {i}/{len(indices)}")

pisos_enriquecidos_Qwen_co_te_ja_ga_pi.to_csv(ruta, sep=";", index=False)
print("✅ Proceso terminado")

El archivo ya existe
Filas a procesar: 276
Procesadas 0/276
Procesadas 100/276
Procesadas 200/276
✅ Proceso terminado


In [ ]:
pisos_enriquecidos_Qwen_co_te_ja_ga['piscina'].value_counts()

,count
piscina,
False,5909
Exterior,2848
Sin piscina,1777
Comunitaria,1475
Climatizada,23


In [ ]:
pisos_enriquecidos_Qwen_co_te_ja_ga_pi['piscina'].value_counts()

,count
piscina,
False,5909
Exterior,2848
Comunitaria,1604
Sin Piscina,1446
Privada,191
Climatizada,34


In [ ]:
import pandas as pd
import torch

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

ruta = "/content/drive/MyDrive/Colab Notebooks/TFM/AñadimosDescripciones/DatasetsEnriquecidos/Qwen/pisos_enriquecidos_Qwen_co_te_ja_ga_pi.csv"

if os.path.exists(ruta):
    print("El archivo ya existe")
    pisos_enriquecidos_Qwen_co_te_ja_ga_pi = pd.read_csv(ruta, sep=";")
else:
    print("El archivo aún no existe")
    pisos_enriquecidos_Qwen_co_te_ja_ga_pi = pisos_enriquecidos_Qwen_co_te_ja_ga.copy()

# =========================
# 3. PROCESAMIENTO
# =========================
indices = pisos_enriquecidos_Qwen_co_te_ja_ga_pi[pisos_enriquecidos_Qwen_co_te_ja_ga_pi["piscina"] == "False"].index

print(f"Filas a procesar: {len(indices)}")
guardar_cada = 100

for i, idx in enumerate(indices):
    descripcion = pisos_enriquecidos_Qwen_co_te_ja_ga_pi.loc[idx, "Descripcion"]

    try:
        nueva_clasificacion = clasifica_piscina_llm(descripcion)
        pisos_enriquecidos_Qwen_co_te_ja_ga_pi.loc[idx, "piscina"] = nueva_clasificacion

    except Exception as e:
        print(f"Error en fila {idx}: {e}")
        continue

    # =========================
    # 4. GUARDADO ITERATIVO
    # =========================
    if i % guardar_cada == 0:
      pisos_enriquecidos_Qwen_co_te_ja_ga_pi.to_csv(ruta, sep=";", index=False)

    if i % guardar_cada == 0:
        print(f"Procesadas {i}/{len(indices)}")

pisos_enriquecidos_Qwen_co_te_ja_ga_pi.to_csv(ruta, sep=";", index=False)
print("✅ Proceso terminado")

El archivo ya existe
Filas a procesar: 5908
Procesadas 0/5908
Procesadas 100/5908
Procesadas 200/5908
Procesadas 300/5908
Procesadas 400/5908
Procesadas 500/5908
Procesadas 600/5908
Procesadas 700/5908
Procesadas 800/5908
Procesadas 900/5908
Procesadas 1000/5908
Procesadas 1100/5908
Procesadas 1200/5908
Procesadas 1300/5908
Procesadas 1400/5908
Procesadas 1500/5908
Procesadas 1600/5908
Procesadas 1700/5908
Procesadas 1800/5908
Procesadas 1900/5908
Procesadas 2000/5908
Procesadas 2100/5908
Procesadas 2200/5908
Procesadas 2300/5908
Procesadas 2400/5908
Procesadas 2500/5908
Procesadas 2600/5908
Procesadas 2700/5908
Procesadas 2800/5908
Procesadas 2900/5908
Procesadas 3000/5908
Procesadas 3100/5908
Procesadas 3200/5908
Procesadas 3300/5908
Procesadas 3400/5908
Procesadas 3500/5908
Procesadas 3600/5908
Procesadas 3700/5908
Procesadas 3800/5908
Procesadas 3900/5908
Procesadas 4000/5908
Procesadas 4100/5908
Procesadas 4200/5908
Procesadas 4300/5908
Procesadas 4400/5908
Procesadas 4500/5908
Pr

In [ ]:
pisos_enriquecidos_Qwen_co_te_ja_ga_pi['piscina'].value_counts()

,count
piscina,
Sin Piscina,7355
Exterior,2848
Comunitaria,1604
Privada,191
Climatizada,34


#### Aire acondicionado

Cargamos el dataset anterior:

In [ ]:
pisos_enriquecidos_Qwen_co_te_ja_ga_pi = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/TFM/AñadimosDescripciones/DatasetsEnriquecidos/Qwen/pisos_enriquecidos_Qwen_co_te_ja_ga_pi.csv', sep=';')

Preparamos el prompt:

In [ ]:
# 1. Definimos la función que habla con el modelo
def clasifica_aire_llm(descripcion):

    prompt = (
        "Analiza la siguiente descripción de una vivienda y clasifica si tiene aire acondicionado. "
        "Responde estrictamente con una de estas 2 opciones:\n"
        "- 'True' (si la vivienda tiene aire acondicionado)\n"
        "- 'False' (si la vivienda no tiene aire acondicionado)\n"
        f"Descripción: {descripcion}"
    )
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    with torch.no_grad(): # Esto ahorra memoria GPU al procesar
        generated_ids = model.generate(**model_inputs, max_new_tokens=10, pad_token_id=tokenizer.eos_token_id)

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    resultado = response.split("assistant\n")[-1].strip().upper()
    # print(resultado)
    # Limpieza rápida por si el modelo añade un punto final o mayúsculas/minúsculas
    resultado_clean = resultado.lower()
    if "true" in resultado_clean:
        return "True"
    elif "false" in resultado_clean:
        return "False Revisada"
    else:
        return "Desconocida Revisada"

Preparamos el codigo para recorrer todas las filas con aire acondicionado:

In [ ]:
import pandas as pd
import torch
import os

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

ruta = "/content/drive/MyDrive/Colab Notebooks/TFM/AñadimosDescripciones/DatasetsEnriquecidos/Qwen/pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai.csv"

if os.path.exists(ruta):
    print("El archivo ya existe")
    pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai = pd.read_csv(ruta, sep=";")
else:
    print("El archivo aún no existe")
    pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai = pisos_enriquecidos_Qwen_co_te_ja_ga_pi.copy()

# =========================
# 3. PROCESAMIENTO
# =========================
indices = pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai[pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai["aire_acondicionado"] == 'False'].index

print(f"Filas a procesar: {len(indices)}")
guardar_cada = 100

for i, idx in enumerate(indices):
    descripcion = pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai.loc[idx, "Descripcion"]

    try:
        nueva_clasificacion = clasifica_aire_llm(descripcion)
        pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai.loc[idx, "aire_acondicionado"] = nueva_clasificacion

    except Exception as e:
        print(f"Error en fila {idx}: {e}")
        continue

    # =========================
    # 4. GUARDADO ITERATIVO
    # =========================
    if i % guardar_cada == 0:
      pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai.to_csv(ruta, sep=";", index=False)

    if i % guardar_cada == 0:
        print(f"Procesadas {i}/{len(indices)}")

pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai.to_csv(ruta, sep=";", index=False)
print("✅ Proceso terminado")

El archivo ya existe
Filas a procesar: 1861
Procesadas 0/1861
Procesadas 100/1861
Procesadas 200/1861
Procesadas 300/1861
Procesadas 400/1861
Procesadas 500/1861
Procesadas 600/1861
Procesadas 700/1861
Procesadas 800/1861
Procesadas 900/1861
Procesadas 1000/1861
Procesadas 1100/1861
Procesadas 1200/1861
Procesadas 1300/1861
Procesadas 1400/1861
Procesadas 1500/1861
Procesadas 1600/1861
Procesadas 1700/1861
Procesadas 1800/1861
✅ Proceso terminado


In [ ]:
pisos_enriquecidos_Qwen_co_te_ja_ga_pi['aire_acondicionado'].value_counts()

,count
aire_acondicionado,
False,8764
True,3268


In [ ]:
pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai['aire_acondicionado'].value_counts()

,count
aire_acondicionado,
False Revisada,8098
True,3934


#### Trastero

Cargamos el dataset anterior:

In [ ]:
pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/TFM/AñadimosDescripciones/DatasetsEnriquecidos/Qwen/pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai.csv', sep=';')

Preparamos el prompt:

In [ ]:
# 1. Definimos la función que habla con el modelo
def clasifica_trastero_llm(descripcion):

    prompt = (
        "Analiza la siguiente descripción de una vivienda y clasifica si tiene trastero. "
        "Responde estrictamente con una de estas 2 opciones:\n"
        "- 'True' (si la vivienda tiene trastero)\n"
        "- 'False' (si la vivienda no tiene trastero)\n"
        f"Descripción: {descripcion}"
    )
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    with torch.no_grad(): # Esto ahorra memoria GPU al procesar
        generated_ids = model.generate(**model_inputs, max_new_tokens=10, pad_token_id=tokenizer.eos_token_id)

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    resultado = response.split("assistant\n")[-1].strip().upper()
    # print(resultado)
    # Limpieza rápida por si el modelo añade un punto final o mayúsculas/minúsculas
    resultado_clean = resultado.lower()
    if "true" in resultado_clean:
        return "True"
    elif "false" in resultado_clean:
        return "False Revisada"
    else:
        return "Desconocida Revisada"

Preparamos el codigo para recorrer todas las filas con piscina:

In [ ]:
import pandas as pd
import torch
import os

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

ruta = "/content/drive/MyDrive/Colab Notebooks/TFM/AñadimosDescripciones/DatasetsEnriquecidos/Qwen/pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai_tr.csv"

if os.path.exists(ruta):
    print("El archivo ya existe")
    pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai_tr = pd.read_csv(ruta, sep=";")
else:
    print("El archivo aún no existe")
    pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai_tr = pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai.copy()

# =========================
# 3. PROCESAMIENTO
# =========================
indices = pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai_tr[pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai_tr["trastero"] == 'False'].index

print(f"Filas a procesar: {len(indices)}")
guardar_cada = 100

for i, idx in enumerate(indices):
    descripcion = pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai_tr.loc[idx, "Descripcion"]

    try:
        nueva_clasificacion = clasifica_trastero_llm(descripcion)
        pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai_tr.loc[idx, "trastero"] = nueva_clasificacion

    except Exception as e:
        print(f"Error en fila {idx}: {e}")
        continue

    # =========================
    # 4. GUARDADO ITERATIVO
    # =========================
    if i % guardar_cada == 0:
      pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai_tr.to_csv(ruta, sep=";", index=False)

    if i % guardar_cada == 0:
        print(f"Procesadas {i}/{len(indices)}")

pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai_tr.to_csv(ruta, sep=";", index=False)
print("✅ Proceso terminado")

El archivo ya existe
Filas a procesar: 5541
Procesadas 0/5541
Procesadas 100/5541
Procesadas 200/5541
Procesadas 300/5541
Procesadas 400/5541
Procesadas 500/5541
Procesadas 600/5541
Procesadas 700/5541
Procesadas 800/5541
Procesadas 900/5541
Procesadas 1000/5541
Procesadas 1100/5541
Procesadas 1200/5541
Procesadas 1300/5541
Procesadas 1400/5541
Procesadas 1500/5541
Procesadas 1600/5541
Procesadas 1700/5541
Procesadas 1800/5541
Procesadas 1900/5541
Procesadas 2000/5541
Procesadas 2100/5541
Procesadas 2200/5541
Procesadas 2300/5541
Procesadas 2400/5541
Procesadas 2500/5541
Procesadas 2600/5541
Procesadas 2700/5541
Procesadas 2800/5541
Procesadas 2900/5541
Procesadas 3000/5541
Procesadas 3100/5541
Procesadas 3200/5541
Procesadas 3300/5541
Procesadas 3400/5541
Procesadas 3500/5541
Procesadas 3600/5541
Procesadas 3700/5541
Procesadas 3800/5541
Procesadas 3900/5541
Procesadas 4000/5541
Procesadas 4100/5541
Procesadas 4200/5541
Procesadas 4300/5541
Procesadas 4400/5541
Procesadas 4500/5541
Pr

In [ ]:
pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai['trastero'].value_counts()

,count
trastero,
False,9942
True,2090


In [ ]:
pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai_tr['trastero'].value_counts()

,count
trastero,
False Revisada,8050
True,3982


#### Ascensor

Cargamos el dataset anterior:

In [ ]:
pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai_tr = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/TFM/AñadimosDescripciones/DatasetsEnriquecidos/Qwen/pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai_tr.csv', sep=';')

Preparamos el prompt:

In [ ]:
# 1. Definimos la función que habla con el modelo
def clasifica_ascensor_llm(descripcion):

    prompt = (
        "Analiza la siguiente descripción de una vivienda y clasifica si tiene ascensor. "
        "Responde estrictamente con una de estas 2 opciones:\n"
        "- 'True' (si la vivienda tiene ascensor)\n"
        "- 'False' (si la vivienda no tiene ascensor)\n"
        f"Descripción: {descripcion}"
    )
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    with torch.no_grad(): # Esto ahorra memoria GPU al procesar
        generated_ids = model.generate(**model_inputs, max_new_tokens=10, pad_token_id=tokenizer.eos_token_id)

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    resultado = response.split("assistant\n")[-1].strip().upper()
    # print(resultado)
    # Limpieza rápida por si el modelo añade un punto final o mayúsculas/minúsculas
    resultado_clean = resultado.lower()
    if "true" in resultado_clean:
        return "True"
    elif "false" in resultado_clean:
        return "False Revisada"
    else:
        return "Desconocida Revisada"

Preparamos el codigo para recorrer todas las filas con piscina:

In [ ]:
import pandas as pd
import torch

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

ruta = "/content/drive/MyDrive/Colab Notebooks/TFM/AñadimosDescripciones/DatasetsEnriquecidos/Qwen/pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai_tr_as.csv"

if os.path.exists(ruta):
    print("El archivo ya existe")
    pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai_tr_as = pd.read_csv(ruta, sep=";")
else:
    print("El archivo aún no existe")
    pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai_tr_as = pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai_tr.copy()

# =========================
# 3. PROCESAMIENTO
# =========================
indices = pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai_tr_as[pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai_tr_as["ascensor"] == 'False'].index

print(f"Filas a procesar: {len(indices)}")
guardar_cada = 100

for i, idx in enumerate(indices):
    descripcion = pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai_tr_as.loc[idx, "Descripcion"]

    try:
        nueva_clasificacion = clasifica_ascensor_llm(descripcion)
        pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai_tr_as.loc[idx, "ascensor"] = nueva_clasificacion

    except Exception as e:
        print(f"Error en fila {idx}: {e}")
        continue

    # =========================
    # 4. GUARDADO ITERATIVO
    # =========================
    if i % guardar_cada == 0:
      pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai_tr_as.to_csv(ruta, sep=";", index=False)

    if i % guardar_cada == 0:
        print(f"Procesadas {i}/{len(indices)}")

pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai_tr_as.to_csv(ruta, sep=";", index=False)
print("✅ Proceso terminado")

El archivo aún no existe
Filas a procesar: 0
✅ Proceso terminado


In [ ]:
pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai_tr['ascensor'].value_counts()

,count
ascensor,
False,9331
True,2701


In [ ]:
pisos_enriquecidos_Qwen_co_te_ja_ga_pi_ai_tr_as['ascensor'].value_counts()

,count
ascensor,
False,9331
True,2701


#### Aire acondicionado (Repetir)

Cargamos el dataset anterior:

In [5]:
pisos_qwen = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/TFM/AñadimosDescripciones/DatasetsEnriquecidos/Qwen/pisos_enriquecidos_Qwen_definitivo.csv', sep=';')

print(pisos_qwen.duplicated().sum())

pisos_qwen['aire_acondicionado'].value_counts()

0


,count
aire_acondicionado,
False,7685
True,4347


Preparamos el prompt:

In [6]:
# 1. Definimos la función que habla con el modelo
def clasifica_aire_llm(descripcion):

    prompt = (
        "Analiza la siguiente descripción de una vivienda y clasifica si tiene aire acondicionado. "
        "Responde estrictamente con una de estas 2 opciones:\n"
        "- 'True' (si la vivienda tiene aire acondicionado)\n"
        "- 'False' (si la vivienda no tiene aire acondicionado)\n"
        f"Descripción: {descripcion}"
    )
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    with torch.no_grad(): # Esto ahorra memoria GPU al procesar
        generated_ids = model.generate(**model_inputs, max_new_tokens=10, pad_token_id=tokenizer.eos_token_id)

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    resultado = response.split("assistant\n")[-1].strip().upper()
    # print(resultado)
    # Limpieza rápida por si el modelo añade un punto final o mayúsculas/minúsculas
    resultado_clean = resultado.lower()
    if "true" in resultado_clean:
        return "True"
    elif "false" in resultado_clean:
        return "False Revisada"
    else:
        return "Desconocida Revisada"

Preparamos el codigo para recorrer todas las filas con aire acondicionado:

In [7]:
import pandas as pd
import torch
import os

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

ruta = "/content/drive/MyDrive/Colab Notebooks/TFM/AñadimosDescripciones/DatasetsEnriquecidos/Qwen/pisos_enriquecidos_Qwen_definitivo2.csv"

if os.path.exists(ruta):
    print("El archivo ya existe")
    pisos_qwen2 = pd.read_csv(ruta, sep=";")
else:
    print("El archivo aún no existe")
    pisos_qwen2 = pisos_qwen.copy()

# =========================
# 3. PROCESAMIENTO
# =========================
indices = pisos_qwen2[pisos_qwen2["aire_acondicionado"] == 'False'].index

print(f"Filas a procesar: {len(indices)}")
guardar_cada = 100

for i, idx in enumerate(indices):
    descripcion = pisos_qwen2.loc[idx, "Descripcion"]

    try:
        nueva_clasificacion = clasifica_aire_llm(descripcion)
        pisos_qwen2.loc[idx, "aire_acondicionado"] = nueva_clasificacion

    except Exception as e:
        print(f"Error en fila {idx}: {e}")
        continue

    # =========================
    # 4. GUARDADO ITERATIVO
    # =========================
    if i % guardar_cada == 0:
      pisos_qwen2.to_csv(ruta, sep=";", index=False)

    if i % guardar_cada == 0:
        print(f"Procesadas {i}/{len(indices)}")

pisos_qwen2.to_csv(ruta, sep=";", index=False)
print("✅ Proceso terminado")

El archivo ya existe
Filas a procesar: 4084
Procesadas 0/4084
Procesadas 100/4084
Procesadas 200/4084
Procesadas 300/4084
Procesadas 400/4084
Procesadas 500/4084
Procesadas 600/4084
Procesadas 700/4084
Procesadas 800/4084
Procesadas 900/4084
Procesadas 1000/4084
Procesadas 1100/4084
Procesadas 1200/4084
Procesadas 1300/4084
Procesadas 1400/4084
Procesadas 1500/4084
Procesadas 1600/4084
Procesadas 1700/4084
Procesadas 1800/4084
Procesadas 1900/4084
Procesadas 2000/4084
Procesadas 2100/4084
Procesadas 2200/4084
Procesadas 2300/4084
Procesadas 2400/4084
Procesadas 2500/4084
Procesadas 2600/4084
Procesadas 2700/4084
Procesadas 2800/4084
Procesadas 2900/4084
Procesadas 3000/4084
Procesadas 3100/4084
Procesadas 3200/4084
Procesadas 3300/4084
Procesadas 3400/4084
Procesadas 3500/4084
Procesadas 3600/4084
Procesadas 3700/4084
Procesadas 3800/4084
Procesadas 3900/4084
Procesadas 4000/4084
✅ Proceso terminado


In [8]:
pisos_qwen['aire_acondicionado'].value_counts()

,count
aire_acondicionado,
False,7685
True,4347


In [9]:
pisos_qwen2['aire_acondicionado'].value_counts()

,count
aire_acondicionado,
True,6787
False Revisada,5245


## EDA

In [3]:
pisos_qwen = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/TFM/AñadimosDescripciones/DatasetsEnriquecidos/Qwen/pisos_enriquecidos_Qwen_definitivo.csv', sep=';')

columnas = [
    "conservacion",
    "terraza",
    "jardin",
    "garaje",
    "piscina",
    "aire_acondicionado",
    "trastero",
    "ascensor",
]

for col in columnas:
    print(f"=== {col.upper()} ===")
    print(pisos_qwen[col].value_counts(dropna=False))
    print("\n")

=== CONSERVACION ===
conservacion
A estrenar              6273
En buen estado          4311
Reformado                933
A reformar               299
Desconocida Revisada     216
Name: count, dtype: int64


=== TERRAZA ===
terraza
True              10129
False Revisada     1903
Name: count, dtype: int64


=== JARDIN ===
jardin
Privado                 7006
Sin Jardín              3766
Comunitario             1254
Desconocida Revisada       6
Name: count, dtype: int64


=== GARAJE ===
garaje
Privado                 5557
Sin Garaje              5478
Opcional                 653
Comunitario              184
Desconocida Revisada     160
Name: count, dtype: int64


=== PISCINA ===
piscina
Exterior                5727
Sin Piscina             3036
Comunitaria             2361
Desconocida Revisada     844
Climatizada               63
Privada                    1
Name: count, dtype: int64


=== AIRE_ACONDICIONADO ===
aire_acondicionado
True              6787
False Revisada    5245
Name: count, d

In [4]:
print(pisos_qwen.duplicated().sum())

0
